# Phase 1 SFT on Modal Notebooks — `post-training-lab`

Same Phase 1 SFT flow as the Kaggle notebook, but on [Modal Notebooks](https://modal.com/notebooks): full SFT of Qwen2.5-0.5B-Instruct on a 5k slice of UltraChat-200k with QLoRA, then re-runs the same eval harness with the trained adapter so `results/metrics.json` ends up with a second row directly comparable to the Phase 0 baseline.

Flow: smoke (50 steps, no push) → full run (~80 steps, pushes adapter to HF Hub) → post-SFT eval (appends `sft_v1` row to `metrics.json`).

**Before you start**
1. Upload this `.ipynb` at [modal.com/notebooks](https://modal.com/notebooks) → **New notebook** → **Upload**.
2. Right sidebar → **Kernel settings**:
   - **GPU** → `A10G` (recommended; ~$1.10/hr). Free $30 signup credit covers ~25 SFT runs.
   - **CPU/memory** → defaults are fine for a 0.5B model.
   - **Idle timeout** → bump to 60+ min so the full run isn't killed mid-train.
3. Right sidebar → **Secrets** → attach two Modal secrets (create them once via `modal secret create` on your laptop, or in the dashboard):
   - `hf-token` exposing `HUGGING_FACE_HUB_TOKEN` — **write scope** required for the adapter push. Mint at `huggingface.co/settings/tokens`.
   - `wandb` exposing `WANDB_API_KEY` — optional, only if you want training logs in W&B.
4. Edit the clone cell's `GITHUB_USER` and (if different) the `output.hub_repo` in `configs/sft_qwen05b.yaml`.

Reference: `PROJECT.md` §5.3 (Modal as the primary paid option), §6 Phase 1 (deliverable + success criterion), §4.4 (HF Hub naming), §5.4 (smoke-test discipline).

## 1. GPU sanity check

Fail fast if no accelerator is attached — SFT on CPU is hours-per-step.

In [ ]:
!nvidia-smi

## 2. Clone the repo

Modal Notebooks gives you a writable container filesystem. We clone into `/root/post-training-lab` and `cd` there for the rest of the run.

In [ ]:
GITHUB_USER = "agaonker"
REPO        = "post-training-lab"
BRANCH      = "main"
REPO_PATH   = f"/root/{REPO}"

import os, shutil
os.chdir("/root")
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
!git clone --depth 1 --branch {BRANCH} https://github.com/{GITHUB_USER}/{REPO}.git {REPO_PATH}
os.chdir(REPO_PATH)
!git log -1 --oneline

## 3. Point HF cache at the workspace

Modal Notebooks' container filesystem persists across cell re-runs within the same kernel session. For cross-session caching (avoiding the 5–15 min model redownload on every cold start), attach a Modal Volume at `/cache` in the kernel sidebar and set `HF_HOME=/cache/hf` instead — same code path.

In [ ]:
os.environ["HF_HOME"] = "/root/hf-cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
print("HF_HOME =", os.environ["HF_HOME"])

## 4. Install the project

Modal Notebooks' default image already has PyTorch / NumPy / Pandas pre-installed; `%uv pip install` is the recommended fast installer in the kernel. This pulls trl, peft, transformers, datasets, accelerate, bitsandbytes (Linux-only — Modal's image is Linux so we get bnb), plus the `[eval]` extra for lm-eval.

In [ ]:
%uv pip install -q -e .[eval]

## 5. Verify secrets are injected

Modal Secrets attached to the kernel show up as env vars. The Kaggle notebook reads them via `kaggle_secrets`; on Modal they're already in `os.environ` — we just sanity-check them and normalize the variable names the training script expects.

If `HF_TOKEN` is missing, the Hub push at the end of `make sft` will fail with 401. Mint a fresh fine-grained token with **write** access to your namespace before running.

In [ ]:
# Modal secret `hf-token` typically exposes HUGGING_FACE_HUB_TOKEN; the
# training script also reads HF_TOKEN. Mirror them so either works.
if "HUGGING_FACE_HUB_TOKEN" in os.environ and "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"]

for key in ("HF_TOKEN", "WANDB_API_KEY"):
    print(f"{key:14s} {'loaded' if os.environ.get(key) else 'NOT SET'}")

if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_MODE"] = "disabled"

## 6. Patch dtype to fp16 on T4 / P100 (only if you picked one)

A10G / A100 / H100 / L4 are native bf16 — leave the config alone. T4 / P100 emulate bf16 in software at ~half speed; for those, patch to fp16. This cell auto-detects and does nothing on bf16-native cards.

In [ ]:
import subprocess, re, pathlib
gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]
).decode().strip()
print("GPU:", gpu_name)

if "T4" in gpu_name or "P100" in gpu_name:
    p = pathlib.Path("configs/base.yaml")
    p.write_text(re.sub(r"dtype:\s*bfloat16", "dtype: float16", p.read_text()))
    p.write_text(
        re.sub(r"bnb_4bit_compute_dtype:\s*bfloat16", "bnb_4bit_compute_dtype: float16", p.read_text())
    )
    print("Patched configs/base.yaml -> dtype: float16 (native fp16 beats emulated bf16).")
else:
    print("Native bf16 GPU — leaving configs untouched.")

## 7. SFT smoke (50 steps, no Hub push)

Catches wiring failures before paying full wall-clock. PROJECT.md §5.4: *"Never launch [a long] run on code you haven't run 50 steps of on free tier."* On A10G with QLoRA, 50 steps is ~2 minutes. If this fails — dataset format error, OOM, peft target_modules mismatch — fix here, not in the full run.

In [ ]:
!make sft-smoke

## 8. Full SFT run + push to HF Hub

5000 UltraChat samples × 1 epoch × effective batch 16 ≈ ~80 steps. A10G wall-clock with QLoRA is roughly 5–10 minutes (vs. 10–20 on T4). On success, the adapter is pushed to `<output.hub_repo>` from `configs/sft_qwen05b.yaml` (default: `agaonker/atlas-sft-qwen05b-v1`).

The summary JSON at the end is the same shape as `run_eval`'s — capture it if the cell output gets truncated.

In [ ]:
!make sft

## 9. Post-SFT eval — append the Phase 1 row to `metrics.json`

Re-runs the same eval harness used for Phase 0, but with `--adapter` pointing at the adapter we just pushed. Same tasks, same metric extraction, same JSON schema — apples-to-apples by construction (the whole reason `atlas.eval.harness` exists as a single entrypoint).

`--name sft_v1 --method sft` distinguishes the row from the `base` row. The harness writes incrementally to `results/metrics.sft_v1.partial.json` and promotes to `metrics.json` only on success, so a mid-run disconnect won't lose completed tasks.

Edit `HUB_ADAPTER` below if you customized `output.hub_repo` in the YAML.

In [ ]:
HUB_ADAPTER = "agaonker/atlas-sft-qwen05b-v1"

!python -m atlas.eval.harness \
    --config configs/baseline.yaml \
    --name sft_v1 --method sft \
    --adapter {HUB_ADAPTER}

## 10. Inspect the updated `metrics.json`

Two rows should now be present: `base` (Phase 0) and `sft_v1` (Phase 1). The Phase 1 success criterion per PROJECT.md §6 is *"SFT model beats base on IFEval by a clear margin"* — concretely, with the Phase 0 numbers in hand: **IFEval prompt-strict > 0.25 and inst-strict > 0.40**.

In [ ]:
import json, pathlib
doc = json.loads(pathlib.Path("results/metrics.json").read_text())
for run in doc["runs"]:
    print(f"\n--- {run['name']} ({run['method']}) ---")
    for k, v in run["metrics"].items():
        if "sample_len" not in k:
            print(f"  {k:50s} {v:.4f}")

## 11. Get `metrics.json` back to your laptop

- **File browser**: left sidebar in Modal Notebooks → navigate to `post-training-lab/results/metrics.json` → right-click → **Download**. Save to your local `results/metrics.json` (overwrites the Phase-0-only file with the new two-row version) and commit.
- **Persistence across kernel restarts**: by default the container filesystem is ephemeral after the kernel stops. To survive a restart, write `results/` to a Modal Volume mounted in the kernel sidebar instead of `/root/post-training-lab/results/`.